In [ ]:
!pip install timm torch torchvision tqdm matplotlib seaborn scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

USE_ZIP  = True
ZIP_PATH = "/content/drive/MyDrive/augment_dataset_cow.zip"

Mounted at /content/drive


In [ ]:
!pip install timm tqdm matplotlib seaborn scikit-learn -q

In [ ]:
# ============================================================
# CELL 4 — Idhae FULL copy paste pannu Colab cell la
# File upload pannanum vendam — direct run aagum!
# ============================================================

import os, zipfile, shutil, random, time, copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# ─── CONFIG ───────────────────────────────────────────────
ZIP_PATH    = "/content/drive/MyDrive/augment_dataset_cow.zip"
EXTRACT_DIR = "/content/cow_extracted"
DATA_DIR    = "/content/cow_split"
SAVE_PATH   = "/content/best_cow_model.pth"
NUM_CLASSES = 12
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 50
LR          = 1e-4
WEIGHT_DECAY= 1e-2
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SEED        = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ─── STEP 1: UNZIP ────────────────────────────────────────
print("\n📦 Unzipping dataset...")
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print(f"✅ Extracted to {EXTRACT_DIR}")

# Auto detect nested folder
entries = os.listdir(EXTRACT_DIR)
if len(entries) == 1 and os.path.isdir(os.path.join(EXTRACT_DIR, entries[0])):
    src_root = os.path.join(EXTRACT_DIR, entries[0])
else:
    src_root = EXTRACT_DIR

# Check for augment_dataset_cow subfolder
aug = os.path.join(EXTRACT_DIR, "augment_dataset_cow")
if os.path.exists(aug):
    src_root = aug

classes = sorted([d for d in os.listdir(src_root)
                  if os.path.isdir(os.path.join(src_root, d))])
print(f"✅ Found {len(classes)} breeds: {classes}")

# ─── STEP 2: SPLIT (70/15/15) ─────────────────────────────
print("\n✂️ Splitting dataset...")
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)

IMAGE_EXTS = ('.jpg','.jpeg','.png','.bmp','.tiff','.webp')
stats = {"train":0,"val":0,"test":0}

for cls in classes:
    cls_src = os.path.join(src_root, cls)
    images  = [f for f in os.listdir(cls_src) if f.lower().endswith(IMAGE_EXTS)]
    random.shuffle(images)
    n = len(images); t = int(n*TRAIN_RATIO); v = int(n*VAL_RATIO)
    split_map = {"train":images[:t],"val":images[t:t+v],"test":images[t+v:]}
    for split, files in split_map.items():
        dst = os.path.join(DATA_DIR, split, cls)
        os.makedirs(dst, exist_ok=True)
        for f in files:
            shutil.copy(os.path.join(cls_src,f), os.path.join(dst,f))
        stats[split] += len(files)
    print(f"  🐄 {cls:30s} Train:{len(split_map['train'])} | Val:{len(split_map['val'])} | Test:{len(split_map['test'])}")

print(f"\n✅ Split done! Train:{stats['train']} | Val:{stats['val']} | Test:{stats['test']}")

# ─── STEP 3: CBAM MODULE ──────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch//r, 1)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(nn.Flatten(),
                                 nn.Linear(ch, mid, bias=False),
                                 nn.ReLU(inplace=True),
                                 nn.Linear(mid, ch, bias=False))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        s = self.sig(self.mlp(self.avg(x)) + self.mlp(self.mx(x)))
        return x * s.view(x.size(0), x.size(1), 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(1, keepdim=True)
        mx, _ = x.max(1, keepdim=True)
        return x * self.sig(self.conv(torch.cat([avg, mx], 1)))

class CBAM(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.ca = ChannelAttention(ch)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

# ─── STEP 4: SWIN + CBAM MODEL ────────────────────────────
class SwinCBAMCow(nn.Module):
    """
    Swin-Tiny + CBAM inserted after Stage1 (Middle of encoder)
    Stage0 → Stage1 → [CBAM] → Stage2 → Stage3 → Head
    CBAM at 14×14 spatial, 384 channels
    """
    def __init__(self, num_classes=12, pretrained=True):
        super().__init__()
        base = timm.create_model('swin_tiny_patch4_window7_224',
                                 pretrained=pretrained, num_classes=0, global_pool='')
        self.patch_embed        = base.patch_embed
        self.absolute_pos_embed = base.absolute_pos_embed
        self.pos_drop           = base.pos_drop
        self.stage0             = base.layers[0]
        self.stage1             = base.layers[1]
        self.stage2             = base.layers[2]
        self.stage3             = base.layers[3]
        self.norm               = base.norm
        self.cbam = CBAM(384)   # After Stage1 → 384ch, 14×14
        self.head = nn.Sequential(
            nn.LayerNorm(768), nn.Dropout(0.3),
            nn.Linear(768, 512), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(512, num_classes))

    def forward(self, x):
        x = self.patch_embed(x)
        if self.absolute_pos_embed is not None:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)
        x = self.stage0(x)              # (B, 784, 192)
        x = self.stage1(x)              # (B, 196, 384) ← Middle

        # ══ CBAM INJECTION ══════════════════════
        B, L, C = x.shape
        H = W = int(L**0.5)
        x2d = x.transpose(1,2).reshape(B,C,H,W)  # → (B,384,14,14)
        x2d = self.cbam(x2d)                       # Channel + Spatial Attn
        x   = x2d.flatten(2).transpose(1,2)        # → (B,196,384)
        # ════════════════════════════════════════

        x = self.stage2(x)              # (B, 49, 768)
        x = self.stage3(x)              # (B, 49, 768)
        x = self.norm(x)
        x = x.mean(dim=1)               # Global Avg Pool → (B,768)
        return self.head(x)             # (B,12)

# ─── STEP 5: DATALOADERS ──────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE+20, IMG_SIZE+20)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
train_ds = datasets.ImageFolder(os.path.join(DATA_DIR,'train'), train_tf)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR,'val'),   val_tf)
test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR,'test'),  val_tf)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"\n✅ DataLoaders ready | Train:{len(train_ds)} Val:{len(val_ds)} Test:{len(test_ds)}")
print(f"   Breeds: {train_ds.classes}")

# ─── STEP 6: BUILD MODEL ──────────────────────────────────
print("\n🧠 Building Swin + CBAM model (pretrained=True)...")
model = SwinCBAMCow(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
print(f"✅ Total params: {total_p:,} | CBAM @ Stage1 output (14×14, 384ch)")

# ─── STEP 7: OPTIMIZER ────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
backbone_p, new_p = [], []
for name, p in model.named_parameters():
    (new_p if any(k in name for k in ['cbam','head']) else backbone_p).append(p)

optimizer = optim.AdamW([
    {'params': backbone_p, 'lr': LR*0.1},
    {'params': new_p,      'lr': LR}
], weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# ─── STEP 8: TRAINING LOOP ────────────────────────────────
def train_epoch(model, loader, crit, opt, scl):
    model.train()
    ls, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out  = model(imgs)
            loss = crit(out, labels)
        scl.scale(loss).backward()
        scl.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scl.step(opt); scl.update()
        ls += loss.item()*imgs.size(0)
        correct += (out.argmax(1)==labels).sum().item()
        total   += labels.size(0)
    return ls/total, correct/total

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval()
    ls, correct, total = 0, 0, 0
    preds_all, labels_all = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = crit(out, labels)
        ls += loss.item()*imgs.size(0)
        p   = out.argmax(1)
        correct += (p==labels).sum().item()
        total   += labels.size(0)
        preds_all.extend(p.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return ls/total, correct/total, preds_all, labels_all

print(f"\n🚀 Training for {NUM_EPOCHS} epochs...\n")
history  = {"tl":[],"ta":[],"vl":[],"va":[]}
best_acc = 0.0

for epoch in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader, criterion, optimizer, scaler)
    vl, va, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()
    history["tl"].append(tl); history["ta"].append(ta)
    history["vl"].append(vl); history["va"].append(va)
    star = " ⭐ BEST!" if va > best_acc else ""
    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
          f"Train Loss:{tl:.4f} Acc:{ta*100:.2f}% | "
          f"Val Loss:{vl:.4f} Acc:{va*100:.2f}% "
          f"({time.time()-t0:.1f}s){star}")
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), SAVE_PATH)

print(f"\n✅ Best Val Acc: {best_acc*100:.2f}% → Saved: {SAVE_PATH}")

# ─── STEP 9: TEST EVAL ────────────────────────────────────
print("\n🧪 Test Evaluation...")
model.load_state_dict(torch.load(SAVE_PATH))
tl, ta, preds, labels = evaluate(model, test_loader, criterion)
print(f"\n🎯 TEST ACCURACY: {ta*100:.2f}%")
print(f"📉 TEST LOSS    : {tl:.4f}\n")
print(classification_report(labels, preds, target_names=train_ds.classes, digits=4))

# ─── STEP 10: PLOTS ───────────────────────────────────────
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
e = range(1,NUM_EPOCHS+1)
ax1.plot(e, history["tl"],'tomato',label='Train'); ax1.plot(e,history["vl"],'steelblue',label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(e,[a*100 for a in history["ta"]],'tomato',label='Train')
ax2.plot(e,[a*100 for a in history["va"]],'steelblue',label='Val')
ax2.set_title('Accuracy (%)'); ax2.legend()
plt.tight_layout(); plt.savefig('/content/training_curves.png',dpi=150); plt.show()

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(13,11))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
            xticklabels=train_ds.classes,yticklabels=train_ds.classes)
plt.title('Confusion Matrix — 12 Cow Breeds')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.savefig('/content/confusion_matrix.png',dpi=150); plt.show()

print("\n🎉 All done! Model + plots saved in /content/")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   🐑🐄 UNIFIED BREED CLASSIFIER — Sheep + Cow                   ║
# ║   Gradio Web App | Swin Transformer + CBAM                      ║
# ║   Sheep: 7 breeds | Cow: 11 breeds                              ║
# ╚══════════════════════════════════════════════════════════════════╝

# ════════════════════════════════════════════════════════════════════
# CELL 1 — Install
# ════════════════════════════════════════════════════════════════════
!pip install gradio timm torch torchvision -q


# ════════════════════════════════════════════════════════════════════
# CELL 2 — Mount Drive
# ════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

# ════════════════════════════════════════════════════════════════════
# CELL 3 — Imports
# ════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import timm
import gradio as gr
from torchvision import transforms
from PIL import Image
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ════════════════════════════════════════════════════════════════════
# CELL 4 — Model Paths & Class Names
# ════════════════════════════════════════════════════════════════════

# ── Model Paths ───────────────────────────────────────────
SHEEP_MODEL_PATH = "/content/drive/MyDrive/sheep_folder_swintransformer/best_model.pth"
COW_MODEL_PATH   = "/content/best_cow_model.pth"
# If cow model also in drive:
# COW_MODEL_PATH = "/content/drive/MyDrive/sheep_folder_swintransformer/best_cow_model.pth"

# ── Class Names ───────────────────────────────────────────
# Sheep — 7 breeds (alphabetical order, same as training)
SHEEP_CLASSES = [
    'Awassi',
    'Deccani',
    'Dorper',
    'Kathiawari',
    'Merino',
    'Rambouillet',
    'Suffolk'
]

# Cow — 11 breeds (alphabetical order, same as training)
COW_CLASSES = [
    'Ayrshire',
    'Brown',
    'Dane',
    'Friesian',
    'Jersey',
    'brahman',
    'cholistani',
    'dhani',
    'kankarej',
    'sahiwal',
    'sibbi'
]

print(f"✅ Sheep breeds ({len(SHEEP_CLASSES)}): {SHEEP_CLASSES}")
print(f"✅ Cow breeds   ({len(COW_CLASSES)}):   {COW_CLASSES}")

# ════════════════════════════════════════════════════════════════════
# CELL 5 — CBAM + Model Architecture
# (Same architecture for BOTH sheep and cow)
# ════════════════════════════════════════════════════════════════════

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False))
        self.ch_sig  = nn.Sigmoid()
        self.sp_conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sp_sig  = nn.Sigmoid()

    def forward(self, x):
        a = self.mlp(self.avg(x))
        m = self.mlp(self.mx(x))
        x = x * self.ch_sig(a + m).view(x.size(0), x.size(1), 1, 1)
        avg2   = x.mean(1, keepdim=True)
        mx2, _ = x.max(1, keepdim=True)
        return x * self.sp_sig(self.sp_conv(torch.cat([avg2, mx2], 1)))


class SwinCBAMClassifier(nn.Module):
    '''
    Unified Swin + CBAM model
    Works for both Sheep (7 breeds) and Cow (11 breeds)
    CBAM inserted at Stage1 output: (B, 28, 28, 192)
    '''
    def __init__(self, num_classes):
        super().__init__()
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=False,
            num_classes=0,
            global_pool=''
        )
        self.cbam = CBAM(channels=192)  # Stage1 = (B,28,28,192)
        self.swin.layers[1].register_forward_hook(self._cbam_hook)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(768),
            nn.Dropout(0.3),
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def _cbam_hook(self, module, input, output):
        is_tuple = isinstance(output, tuple)
        x    = output[0] if is_tuple else output
        rest = output[1:] if is_tuple else None
        B, H, W, C = x.shape
        x = x.permute(0, 3, 1, 2)
        x = self.cbam(x)
        x = x.permute(0, 2, 3, 1)
        return (x,) + rest if rest else x

    def forward(self, x):
        x = self.swin(x)               # (B, 7, 7, 768)
        x = x.permute(0, 3, 1, 2)     # (B, 768, 7, 7)
        x = self.pool(x)               # (B, 768, 1, 1)
        x = self.head(x)               # (B, num_classes)
        return x

print("✅ SwinCBAMClassifier defined!")

# ════════════════════════════════════════════════════════════════════
# CELL 6 — Load Both Models
# ════════════════════════════════════════════════════════════════
def load_model(path, num_classes, device):
    model = SwinCBAMClassifier(num_classes=num_classes)
    state = torch.load(path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.to(device)
    model.eval()
    return model

print("🐑 Loading Sheep model...")
sheep_model = load_model(SHEEP_MODEL_PATH, len(SHEEP_CLASSES), DEVICE)
print(f"✅ Sheep model loaded! ({len(SHEEP_CLASSES)} breeds)")

print("🐄 Loading Cow model...")
cow_model = load_model(COW_MODEL_PATH, len(COW_CLASSES), DEVICE)
print(f"✅ Cow model loaded! ({len(COW_CLASSES)} breeds)")


# ════════════════════════════════════════════════════════════════════
# CELL 7 — Image Transform
# ════════════════════════════════════════════════════════════════════
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def preprocess(image):
    '''PIL Image → model input tensor'''
    if image.mode != 'RGB':
        image = image.convert('RGB')
    tensor = transform(image).unsqueeze(0).to(DEVICE)  # (1,3,224,224)
    return tensor

print("✅ Transform ready!")

# ════════════════════════════════════════════════════════════════════
# CELL 8 — Prediction Function
# ════════════════════════════════════════════════════════════════════

@torch.no_grad()
def predict(image, animal_type):
    '''
    Main prediction function for Gradio
    animal_type: "Sheep 🐑" or "Cow 🐄"
    '''
    if image is None:
        return "❌ Please upload an image!", {}

    animal_type = animal_type.strip().lower()

    # Select model & classes based on user choice
    if "sheep" in animal_type:
        model   = sheep_model
        classes = SHEEP_CLASSES
        emoji   = "🐑"
        label   = "Sheep"
    elif "cow" in animal_type:
        model   = cow_model
        classes = COW_CLASSES
        emoji   = "🐄"
        label   = "Cow"
    else:
        return "❌ Please select Sheep or Cow!", {}

    # Preprocess & predict
    tensor  = preprocess(image)
    outputs = model(tensor)
    probs   = torch.softmax(outputs, dim=1)[0]

    # Top prediction
    top_idx   = probs.argmax().item()
    top_breed = classes[top_idx]
    top_conf  = probs[top_idx].item() * 100

    # All class probabilities for Gradio label component
    prob_dict = {cls: float(probs[i]) for i, cls in enumerate(classes)}

    result = f"{emoji} **{label} Breed Detected!**\n\n" \
             f"🏆 **Breed : {top_breed}**\n" \
             f"📊 **Confidence : {top_conf:.2f}%**"

    return result, prob_dict

print("✅ predict() function ready!")

# ════════════════════════════════════════════════════════════════════
# CELL 9 — Gradio Web App
# ════════════════════════════════════════════════════════════════════
def gradio_app():
    with gr.Blocks(
        title="🐑🐄 Livestock Breed Classifier",
        theme=gr.themes.Soft()
    ) as app:

        # ── Header ───────────────────────────────────────
        gr.Markdown('''
        # 🐑🐄 Livestock Breed Classifier
        ### Swin Transformer + CBAM | Deep Learning
        Upload an image and select the animal type to identify the breed!
        ''')

        with gr.Row():
            # ── Left Column — Inputs ──────────────────────
            with gr.Column(scale=1):
                image_input = gr.Image(
                    type="pil",
                    label="📸 Upload Image",
                    height=300
                )
                animal_radio = gr.Radio(
                    choices=["Sheep 🐑", "Cow 🐄"],
                    label="🔍 Select Animal Type",
                    value="Cow 🐄"
                )
                predict_btn = gr.Button(
                    "🔎 Classify Breed",
                    variant="primary",
                    size="lg"
                )

            # ── Right Column — Outputs ─────────────────────
            with gr.Column(scale=1):
                result_text = gr.Markdown(
                    label="📋 Result",
                    value="*Result will appear here...*"
                )
                prob_output = gr.Label(
                    label="📊 Breed Probabilities",
                    num_top_classes=5
                )

        # ── Info Section ──────────────────────────────────
        with gr.Accordion("ℹ️ About this App", open=False):
            gr.Markdown(f'''
            **Model Architecture:** Swin Transformer Tiny + CBAM (Convolutional Block Attention Module)

            **CBAM Position:** Inserted at Stage 1 output (Middle of Swin encoder) for enhanced feature attention

            **🐑 Sheep Breeds ({len(SHEEP_CLASSES)}):**
            {", ".join(SHEEP_CLASSES)}

            **🐄 Cow Breeds ({len(COW_CLASSES)}):**
            {", ".join(COW_CLASSES)}

            **How to use:**
            1. Upload a clear image of the animal
            2. Select "Sheep 🐑" or "Cow 🐄"
            3. Click "Classify Breed"
            ''')

        # ── Examples ──────────────────────────────────────
        gr.Markdown("### 📌 How to use:")
        gr.Markdown("""
        1. 📸 **Upload** a sheep or cow image
        2. ☑️ **Select** Sheep or Cow
        3. 🔎 **Click** Classify Breed
        4. 🏆 **See** the breed & confidence score!
        """)

        # ── Button Click ──────────────────────────────────
        predict_btn.click(
            fn=predict,
            inputs=[image_input, animal_radio],
            outputs=[result_text, prob_output]
        )

        # Also predict on image upload (auto)
        image_input.change(
            fn=predict,
            inputs=[image_input, animal_radio],
            outputs=[result_text, prob_output]
        )

    return app


# Launch!
app = gradio_app()
app.launch(
    share=True,           # Public link generate aagum
    debug=False,
    show_error=True
)

Mounted at /content/drive
✅ Device: cuda
✅ Sheep breeds (7): ['Awassi', 'Deccani', 'Dorper', 'Kathiawari', 'Merino', 'Rambouillet', 'Suffolk']
✅ Cow breeds   (11):   ['Ayrshire', 'Brown', 'Dane', 'Friesian', 'Jersey', 'brahman', 'cholistani', 'dhani', 'kankarej', 'sahiwal', 'sibbi']
✅ SwinCBAMClassifier defined!
🐑 Loading Sheep model...
✅ Sheep model loaded! (7 breeds)
🐄 Loading Cow model...


FileNotFoundError: [Errno 2] No such file or directory: '/content/best_cow_model.pth'

In [ ]:
SHEEP_MODEL_PATH = "/content/drive/MyDrive/sheep_folder_swintransformer/best_model.pth"
COW_MODEL_PATH   = "/content/drive/MyDrive/sheep_folder_swintransformer/best_cow_model.pth"

SHEEP_CLASSES = ['Awassi', 'Deccani', 'Dorper', 'Kathiawari',
                 'Merino', 'Rambouillet', 'Suffolk']

COW_CLASSES = ['Ayrshire', 'Brown', 'Dane', 'Friesian', 'Jersey',
               'brahman', 'cholistani', 'dhani', 'kankarej', 'sahiwal', 'sibbi']

print(f"✅ Sheep: {len(SHEEP_CLASSES)} breeds")
print(f"✅ Cow  : {len(COW_CLASSES)} breeds")

✅ Sheep: 7 breeds
✅ Cow  : 11 breeds


In [ ]:
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False))
        self.ch_sig  = nn.Sigmoid()
        self.sp_conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sp_sig  = nn.Sigmoid()

    def forward(self, x):
        a = self.mlp(self.avg(x))
        m = self.mlp(self.mx(x))
        x = x * self.ch_sig(a + m).view(x.size(0), x.size(1), 1, 1)
        avg2   = x.mean(1, keepdim=True)
        mx2, _ = x.max(1, keepdim=True)
        return x * self.sp_sig(self.sp_conv(torch.cat([avg2, mx2], 1)))


class SwinCBAMClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=False, num_classes=0, global_pool='')
        self.cbam = CBAM(channels=192)
        self.swin.layers[1].register_forward_hook(self._cbam_hook)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.LayerNorm(768), nn.Dropout(0.3),
            nn.Linear(768, 512), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(512, num_classes))

    def _cbam_hook(self, module, input, output):
        is_tuple = isinstance(output, tuple)
        x    = output[0] if is_tuple else output
        rest = output[1:] if is_tuple else None
        B, H, W, C = x.shape
        x = x.permute(0,3,1,2)
        x = self.cbam(x)
        x = x.permute(0,2,3,1)
        return (x,) + rest if rest else x

    def forward(self, x):
        x = self.swin(x)
        x = x.permute(0,3,1,2)
        x = self.pool(x)
        return self.head(x)

print("✅ Model defined!")

✅ Model defined!


In [ ]:
def load_model(path, num_classes, device):
    model = SwinCBAMClassifier(num_classes=num_classes)
    state = torch.load(path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.to(device).eval()
    return model

print("🐑 Loading Sheep model...")
sheep_model = load_model(SHEEP_MODEL_PATH, len(SHEEP_CLASSES), DEVICE)
print(f"✅ Sheep loaded!")

print("🐄 Loading Cow model...")
cow_model = load_model(COW_MODEL_PATH, len(COW_CLASSES), DEVICE)
print(f"✅ Cow loaded!")

🐑 Loading Sheep model...
✅ Sheep loaded!
🐄 Loading Cow model...
✅ Cow loaded!


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

@torch.no_grad()
def predict(image, animal_type):
    if image is None:
        return "❌ Please upload an image!", {}
    if "sheep" in animal_type.lower():
        model, classes, emoji = sheep_model, SHEEP_CLASSES, "🐑"
    else:
        model, classes, emoji = cow_model, COW_CLASSES, "🐄"

    img = image.convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    probs  = torch.softmax(model(tensor), dim=1)[0]

    top_idx  = probs.argmax().item()
    top_breed= classes[top_idx]
    top_conf = probs[top_idx].item() * 100
    prob_dict= {cls: float(probs[i]) for i, cls in enumerate(classes)}

    result = f"## {emoji} Detected!\n\n**Breed : {top_breed}**\n\n**Confidence : {top_conf:.2f}%**"
    return result, prob_dict


# Gradio App
with gr.Blocks(title="🐑🐄 Livestock Breed Classifier",
               theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🐑🐄 Livestock Breed Classifier\n### Swin Transformer + CBAM")

    with gr.Row():
        with gr.Column():
            img_input   = gr.Image(type="pil", label="📸 Upload Image", height=300)
            animal_type = gr.Radio(choices=["Sheep 🐑","Cow 🐄"],
                                   value="Cow 🐄", label="Select Animal")
            btn = gr.Button("🔎 Classify", variant="primary", size="lg")

        with gr.Column():
            result_md = gr.Markdown("*Result appears here...*")
            prob_out  = gr.Label(label="📊 Breed Probabilities", num_top_classes=5)

    btn.click(fn=predict, inputs=[img_input, animal_type],
              outputs=[result_md, prob_out])
    img_input.change(fn=predict, inputs=[img_input, animal_type],
                     outputs=[result_md, prob_out])

app.launch(share=True)

/tmp/ipython-input-1346244222.py:30: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🐑🐄 Livestock Breed Classifier",


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57ffa319ccc93fddaa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
